<a href="https://colab.research.google.com/github/zhetiii/KT_3_Pytest-Parametrize/blob/main/%D0%A2%D0%9F%D0%A2_%D0%9A%D0%A23_3%D0%9F3_%D0%94%D0%B8%D0%BA%D0%BE%D0%BB%D0%B5%D0%BD%D0%BA%D0%BE%D0%95%D0%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pytest ipytest

In [4]:
import sqlite3
import pytest
import ipytest

ipytest.autoconfig()

# =====================================================================
# Задание 1: Проверка четности целого числа (is_even)
# =====================================================================

def is_even(number: int) -> bool:
    if type(number) is not int:
        raise TypeError("Параметр должен иметь тип int")
    return number % 2 == 0

@pytest.mark.parametrize("val, expected_result", [
    (12, True),
    (-8, True),
    (0, True),
    (7, False),
    (-15, False),
    (99, False),
], ids=["pos_even", "neg_even", "zero", "pos_odd", "neg_odd", "large_odd"])
def test_is_even_valid_cases(val, expected_result):
    assert is_even(val) is expected_result

@pytest.mark.parametrize("bad_arg", [
    "12", 4.0, True, False, [10], (2, 4), None,
], ids=["str", "float", "bool_true", "bool_false", "list", "tuple", "none"])
def test_is_even_type_errors(bad_arg):
    with pytest.raises(TypeError):
        is_even(bad_arg)

# =====================================================================
# Задание 2: Площадь прямоугольника (calculate_area)
# =====================================================================

def calculate_area(length: float, width: float) -> float:
    if length < 0 or width < 0:
        raise ValueError("Длина и ширина не могут быть меньше нуля")
    return float(length * width)

@pytest.mark.parametrize("l, w, expected_area", [
    (10.0, 5.0, 50.0),
    (0.0, 8.0, 0.0),
    (12.5, 0.0, 0.0),
    (2.5, 4.0, 10.0),
    (1.5, 1.5, 2.25),
], ids=["standard", "zero_len", "zero_width", "floats", "square"])
def test_calculate_area_correct_values(l, w, expected_area):
    assert calculate_area(l, w) == pytest.approx(expected_area)

@pytest.mark.parametrize("invalid_l, invalid_w", [
    (-1.0, 5.0),
    (4.0, -2.5),
    (-10.0, -10.0),
], ids=["neg_length", "neg_width", "both_neg"])
def test_calculate_area_raises_value_error(invalid_l, invalid_w):
    with pytest.raises(ValueError):
        calculate_area(invalid_l, invalid_w)

# =====================================================================
# Задание 3: Определение типа треугольника (classify_triangle)
# =====================================================================

def classify_triangle(a: float, b: float, c: float) -> str:
    sides = sorted([a, b, c])
    if sides[0] <= 0:
        raise ValueError("Стороны должны быть строго больше нуля")
    if sides[0] + sides[1] <= sides[2]:
        raise ValueError("Сумма двух меньших сторон должна быть больше третьей")

    distinct_sides_count = len(set(sides))
    if distinct_sides_count == 1:
        return "равносторонний"
    elif distinct_sides_count == 2:
        return "равнобедренный"
    return "разносторонний"

@pytest.mark.parametrize("side1, side2, side3, expected_category", [
    (7, 7, 7, "равносторонний"),
    (4.5, 4.5, 4.5, "равносторонний"),
    (6, 6, 4, "равнобедренный"),
    (8, 5, 8, "равнобедренный"),
    (3, 7, 7, "равнобедренный"),
    (5, 12, 13, "разносторонний"),
    (7, 8, 9, "разносторонний"),
], ids=["eq_int", "eq_float", "iso_1", "iso_2", "iso_3", "scalene_right", "scalene_std"])
def test_classify_triangle_valid_types(side1, side2, side3, expected_category):
    assert classify_triangle(side1, side2, side3) == expected_category

@pytest.mark.parametrize("s1, s2, s3", [
    (0, 4, 4),
    (-5, 5, 5),
    (2, 2, 0),
    (2, 3, 5),
    (10, 2, 3),
    (1, 15, 2),
], ids=["zero_side", "neg_side", "zero_last", "sum_equal", "sum_less_1", "sum_less_2"])
def test_classify_triangle_invalid_geometry(s1, s2, s3):
    with pytest.raises(ValueError):
        classify_triangle(s1, s2, s3)

# =====================================================================
# Задание 4: Доработка решений из КТ-2
# =====================================================================

def get_string_length(text: str) -> int:
    return len(text)

@pytest.mark.parametrize("sample_text, expected_len", [
    ("", 0),
    ("   ", 3),
    ("Hello World!", 12),
    ("Строка1\nСтрока2", 15),
    ("Python_3.11", 11),
], ids=["empty", "spaces", "ascii", "multiline", "tech_str"])
def test_get_string_length_parametrized(sample_text, expected_len):
    assert get_string_length(sample_text) == expected_len

def save_string_to_file(filepath: str, text: str) -> None:
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(text)

@pytest.mark.parametrize("fname, text_content", [
    ("test_regular.txt", "Основной текст для файла"),
    ("test_empty.txt", ""),
    ("test_symbols.txt", "!@#$%^&*()_+-= ~`"),
], ids=["regular", "empty", "symbols"])
def test_save_string_to_file_parametrized(tmp_path, fname, text_content):
    out_file = tmp_path / fname
    save_string_to_file(str(out_file), text_content)
    assert out_file.is_file()
    assert out_file.read_text(encoding="utf-8") == text_content

@pytest.fixture
def db_session():
    connection = sqlite3.connect(":memory:")
    cur = connection.cursor()
    cur.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT);")
    connection.commit()
    yield connection
    cur.execute("DROP TABLE IF EXISTS users;")
    connection.commit()
    connection.close()

@pytest.mark.parametrize("user_id, name_val", [
    (1, "Евгения"),
    (2, "Алексей"),
    (100, "Анна"),
], ids=["id_1", "id_2", "id_100"])
def test_sqlite_insert_parametrized(db_session, user_id, name_val):
    cursor = db_session.cursor()
    cursor.execute("INSERT INTO users (id, name) VALUES (?, ?);", (user_id, name_val))
    db_session.commit()
    cursor.execute("SELECT name FROM users WHERE id = ?;", (user_id,))
    record = cursor.fetchone()
    assert record is not None
    assert record[0] == name_val

In [5]:
ipytest.run("-v")

======================================= test session starts ========================================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 45 items

t_024a25dcaa104ba69042eb6fb210aa05.py .............................................          [100%]

======================================== 45 passed in 0.18s ========================================


<ExitCode.OK: 0>